# 02 - From scanner findings to tasks

**Purpose:** Turn merged scanner findings into per-service migration tasks that keep finding IDs as evidence.

**Inputs:** demo_bank_app/configs (SYNTHETIC) via cryptoshift.planning

**Outputs:** findings table, findings-per-service chart, evidence-source chart, task table

**Status labels:** VERIFIED on SYNTHETIC configs. Unknown purpose = HOLD (never a PQC mapping).

Context: the demo bank in `demo_bank_app/` is SYNTHETIC. Charts are also saved to `scratch/derived/member3/figs/`.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys, json, subprocess
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "cryptoshift").is_dir())
for p in (str(ROOT / "src"), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
DERIVED = ROOT / "scratch" / "derived" / "member3"
FIGS = DERIVED / "figs"
FIGS.mkdir(parents=True, exist_ok=True)
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
PY = sys.executable

def ensure(rel_path, script, *args):
    """Run a project script once if its output file is missing (keeps notebooks runnable in a fresh checkout)."""
    target = ROOT / rel_path
    if not target.is_file():
        print(f"{rel_path} missing -> running {script} ...")
        subprocess.run([PY, str(ROOT / "scripts" / script), *args], cwd=ROOT, check=True)
    return target

def savefig(name):
    plt.tight_layout(); plt.savefig(FIGS / f"{name}.png", dpi=120); plt.show()
# This notebook reads no credentials. If one is ever needed, read it from an environment variable at runtime and
# never display it.

In [ ]:
from cryptoshift.planning.planner import build_problem
p = build_problem()
find = pd.DataFrame([{k: f.to_dict()[k] for k in ('finding_id','service_id','file_path','line_number','legacy_algorithm','operation_purpose','detector')} for f in p['findings']])
find

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
pd.crosstab(find['service_id'], find['operation_purpose']).plot(kind='barh', stacked=True, ax=ax[0], colormap='tab10', title='Findings per service by operation purpose')
ax[0].set_xlabel('findings')
ev = pd.Series({s: v['evidence_source'] for s, v in p['services'].items()}).value_counts()
ax[1].pie(ev.values, labels=ev.index, autopct='%1.0f%%', colors=['#2a9d8f', '#e9c46a', '#9e9e9e'][:len(ev)])
ax[1].set_title('Services by evidence source')
savefig('02_findings_and_evidence')

In [ ]:
pd.DataFrame([{'service': s, 'evidence': v['evidence_source'], 'findings': len(v['finding_ids']), 'holds': v['hold_finding_ids'], 'criticality (source)': f"{v['criticality']} ({v['field_sources'].get('criticality','').split(' ')[0]})", 'candidates': v['candidate_targets']} for s, v in p['services'].items()])

`DECLARED_ONLY` services (ledger_audit, settlement_batch) have no scanned files; their algorithms come from the synthetic descriptor and stay on HOLD.